导入依赖

In [1]:
import torch
import torchvision
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from models.ShuffleNetV2 import ShuffleNetV2
from models.ShuffleNetV2_exp_channel_shuffle import ShuffleNetV2 as ShuffleNetV2_expCS

设置随机种子

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果使用多GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 100
# 在训练代码开头调用
set_seed(seed) # 42, 2025, 1024, 512, 100

导入数据集

In [3]:
def preprocessing(tensor):
    if tensor.mean() > 0.5:
        tensor = 1 - tensor
    return tensor

In [4]:
def tensor_transform(x):
    if x.shape[1] == 1:
        return x.repeat_interleave(3, dim= 1)
    elif x.shape[1] == 4:
        return x[:, :3, :, :]
    elif x.shape[1] == 3:
        return x
    else:
        raise ValueError('Invalid input shape')

tensor_rgb_transform = transforms.Lambda(tensor_transform)

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(preprocessing),
    transforms.Normalize((0.1307, ), (0.3081, )),
    transforms.Grayscale(1)
])

In [6]:
mnist_db = datasets.MNIST(root= 'dataFolder/', train= True, transform= transform, download= True)
mnist_loader = DataLoader(mnist_db, batch_size= 128, shuffle= True)

In [7]:
our_db = datasets.ImageFolder(root= 'dataFolder/num', transform= transform)
our_loader = DataLoader(our_db, batch_size= 128, shuffle= True)

模型训练

In [8]:
def train_model(model, train_loader, in_ch):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr= 0.001)
    model.train()

    losses = []
    for epoch in range(10):  # 训练10个周期
        correct = 0
        total = 0
        for idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            if in_ch == 3: x = tensor_rgb_transform(x)
            out = model(x)
            loss = criterion(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += x.shape[0]
            correct += (torch.argmax(out, dim= -1) == y).sum().item()
            losses.append(loss.item())
            if idx % 1000 == 0:
                print('Epoch: {}, Loss: {}'.format(epoch, loss.item()))
                print('Accuracy: {:.2f}%'.format(correct * 100 / total))

    return model.state_dict(), losses

In [ ]:
model_1C = ShuffleNetV2(n_classes= 10, in_ch= 1)
state_dict, losses = train_model(model_1C, mnist_loader, in_ch= 1)
torch.save(state_dict, f'save_models/shufV2/ShuffleNetV2_1C_MD_{seed}.pth')
print('ShuffleNetV2_1C saved!')

model_3C = ShuffleNetV2(n_classes= 10, in_ch= 3)
state_dict, losses = train_model(model_3C, mnist_loader, in_ch= 3)
torch.save(state_dict, f'save_models/shufV2/ShuffleNetV2_3C_MD_{seed}.pth')
print('ShuffleNetV2_3C saved!')

model_1C_expSE = ShuffleNetV2_expCS(n_classes= 10, in_ch= 1)
state_dict, losses = train_model(model_1C_expSE, mnist_loader, in_ch= 1)
torch.save(state_dict, f'save_models/shufV2/ShuffleNetV2_expCS_1C_MD_{seed}.pth')
print('ShuffleNetV2_expCS_1C saved!')

model_3C_expSE = ShuffleNetV2_expCS(n_classes= 10, in_ch= 3)
state_dict, losses = train_model(model_3C_expSE, mnist_loader, in_ch= 3)
torch.save(state_dict, f'save_models/shufV2/ShuffleNetV2_expCS_3C_MD_{seed}.pth')
print('ShuffleNetV2_expCS_3C saved!')

# model_1C = ShuffleNetV2(n_classes= 10, in_ch= 1)
# state_dict, losses = train_model(model_1C, our_loader, in_ch= 1)
# torch.save(state_dict, 'save_models/OD/ShuffleNetV2_1C_OD_42.pth')
# print('ShuffleNetV2_1C saved!')

# model_3C = ShuffleNetV2(n_classes= 10, in_ch= 3)
# state_dict, losses = train_model(model_3C, our_loader, in_ch= 3)
# torch.save(state_dict, 'save_models/OD/ShuffleNetV2_3C_OD_42.pth')
# print('ShuffleNetV2_3C saved!')

model size is 1.0x
Epoch: 0, Loss: 2.306122064590454
Accuracy: 8.59%
